# **Synthetic Dataset Creation using Faker Library**

# 1. Synthetic Dataset Creation

In [1]:
# installing faker for the generation of synthetic dataset
!pip install faker

# importing libraries
import pandas as pd
import numpy as np
from datetime import datetime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 27.8 MB/s eta 0:00:00


In [2]:
# importing the Faker class from the faker library
from faker import Faker

# Creating an instance of Faker class
fake_generator = Faker()

# Usig seed for reproducibility of generated records
fake_generator.seed_instance(42)

In [3]:
dataset_count = 1000   # Number of records to be generated

# Names of columns of synthetic Dataset
dataset_cols = ['Applicant_ID','Applicant_Name','Email','Phone','Domain_Applied','University','Application_Date','Status']

schema = {
    'Applicant_ID': str,
    'Applicant_Name': str,
    'Email': str,
    'Phone': str,
    'Domain_Applied': str,
    'University': str,
    'Application_Date': datetime,
    'Status': str
}

# user defined columns
user_defined_cols = {
    'Domain_Applied': ['Machine Learning', 'Data Science', 'Artificial Intelligence', 'Deep Learning', 'Computer Vision','Web Development'],
    'University':['UET'],
    'Status': ['Under Review', 'Selected', 'Rejected']
    }

In [4]:
applicants = []
for i in range(1, dataset_count+1):
    applicant = {
        'Applicant_ID': f"APP{i:04d}",
        'Applicant_Name': fake_generator.name(),
        'Email': fake_generator.email(),
        'Phone': fake_generator.phone_number(),
        'Domain_Applied': fake_generator.random_element(user_defined_cols['Domain_Applied']),
        'University': fake_generator.random_element(user_defined_cols['University']),
        'Application_Date': fake_generator.date_between(start_date='-1y', end_date='today'),
        'Status': fake_generator.random_element(user_defined_cols['Status'])
    }

    applicants.append(applicant)

In [5]:
df = pd.DataFrame(applicants, columns = dataset_cols)
df.to_csv("applicants.csv",index = False)
df.head()


,Applicant_ID,Applicant_Name,Email,Phone,Domain_Applied,University,Application_Date,Status
0,APP0001,Allison Hill,donaldgarcia@example.net,+1-219-560-0133,Web Development,UET,2026-01-02,Rejected
1,APP0002,Sean Blake,helenpeterson@example.org,651.216.1559,Web Development,UET,2026-04-28,Selected
2,APP0003,Edward Fuller,barbara10@example.net,441.731.6475,Artificial Intelligence,UET,2026-06-16,Rejected
3,APP0004,Maria Lynch,amandasanchez@example.com,(748)535-0305x6413,Web Development,UET,2026-03-07,Rejected
4,APP0005,Derek Wright,julie69@example.com,(332)887-1012x269,Deep Learning,UET,2026-04-25,Rejected


# **Making the Synthetic Data Inconsistent**

**copy of the generated dataset**


In [6]:
# Making the copy of the generated dataset so the original data is not affected
df_messy = df.copy()

**Adding Missingness and Whitespaces in Applicant Names**



In [7]:
# Introducing missing values in Applicant Name column
n_missing = int(0.05 * len(df_messy))

np.random.seed(42)

missing_indices = np.random.choice(
    df_messy.index,
    n_missing,
    replace=False
)

# Checking if missingness is added
df_messy.loc[missing_indices, 'Applicant_Name'] = np.nan
df_messy["Applicant_Name"].isna().sum()

np.int64(50)

In [8]:
# Adding leading and training whitespaces

n_whitespace = int(0.10 * len(df_messy))
np.random.seed(42)

whitespace_indices = np.random.choice(
    df_messy.index,
    n_whitespace,
    replace = False
)

for idx in whitespace_indices:
  if pd.notna(df_messy.loc[idx, "Applicant_Name"]):
    df_messy.loc[idx ,"Applicant_Name"] = (
        " " + df_messy.loc[idx, "Applicant_Name"] + " "
    )


In [9]:
# Checking whitespaces
for name in df_messy["Applicant_Name"].dropna():
  if name != name.strip():
    print(repr(name))
    break

' Carmen Smith '


**Adding Missingness and Casing Issues in Emails**

In [10]:
# Introducing missing values in Email column
n_missing_emails = int(0.15 * len(df_messy))

np.random.seed(42)

missing_email_indices = np.random.choice(
    df_messy.index,
    n_missing_emails,
    replace=False
)

# Checking if missingness is added
df_messy.loc[missing_email_indices, 'Email'] = np.nan
df_messy['Email'].isna().sum()

np.int64(150)

In [11]:
from numpy._core.defchararray import capitalize
from numpy.random import random
# Adding Casing issues
n_mixed_emails = int(0.10 * len(df_messy))

email_case_indices = np.random.choice(
    df_messy.index,
    n_mixed_emails,
    replace = False
)

for idx in email_case_indices:
  email = df_messy.loc[idx, "Email"]

  if pd.notna(email):
    df_messy.loc[idx, "Email"] = (
        email.upper() if np.random.random() < 0.5 else email.capitalize()
    )

df_messy['Email'].head(10)


,Email
0,donaldgarcia@example.net
1,helenpeterson@example.org
2,barbara10@example.net
3,amandasanchez@example.com
4,julie69@example.com
5,kayla51@example.com
6,john39@example.org
7,wrightcaleb@example.org
8,esanchez@example.com
9,DWHITE@EXAMPLE.ORG


**Introducing Inconsistent casing/spelling in Domain Applied column**

In [12]:
domain_variations = {
    "Web Development": ["web dev", "WEB DEV", "Web development"],
    "Data Science": ["data science", "DATA SCIENCE", "Data science"],
    "Artificial Intelligence": ["AI", "artificial intelligence", "ARTIFICIAL INTELLIGENCE"]
}

n_messy_domains = int(0.20 * len(df_messy))

domain_indices = np.random.choice(
    df_messy.index,
    size=n_messy_domains,
    replace=False
)

# Replace selected domain values with inconsistent versions
for idx in domain_indices:
    domain = df_messy.loc[idx, "Domain_Applied"]

    if pd.notna(domain) and domain in domain_variations:
        df_messy.loc[idx, "Domain_Applied"] = np.random.choice(
            domain_variations[domain]
        )

print(df_messy["Domain_Applied"].value_counts())

Domain_Applied
Deep Learning              182
Machine Learning           163
Computer Vision            162
Data Science               146
Web Development            133
Artificial Intelligence    111
Web development             16
ARTIFICIAL INTELLIGENCE     15
data science                14
WEB DEV                     12
artificial intelligence     10
DATA SCIENCE                10
web dev                      9
AI                           9
Data science                 8
Name: count, dtype: int64


**Introducing some blank, some inconsistent abbreviations in University Column**

In [13]:
# Map standard university names to inconsistent versions
university_variations = {
    "UET": ["University of Engineering and Technology", "U.E.T", "uet", "Uet lahore"]
}

n_messy_universities = int(0.30 * len(df_messy))

university_indices = np.random.choice(
    df_messy.index,
    size=n_messy_universities,
    replace=False
)

# Introduce inconsistent university names
for idx in university_indices:
    university = df_messy.loc[idx, "University"]

    if pd.notna(university) and university in university_variations:
        df_messy.loc[idx, "University"] = np.random.choice(
            university_variations[university]
        )


In [14]:
# Introducing missing values
n_missing_universities = int(0.35 * len(df_messy))

np.random.seed(42)
missing_university_indices = np.random.choice(
    df_messy.index,
    size=n_missing_universities,
    replace=False
)

df_messy.loc[
    missing_university_indices, "University"
] = np.nan

In [15]:
# Checking if variations are introduced
print(df_messy["University"].value_counts())

University
UET                                         452
uet                                          56
University of Engineering and Technology     50
Uet lahore                                   46
U.E.T                                        46
Name: count, dtype: int64


**Inconsistent date formats**

In [16]:

# Select 10% of rows to change date formats
n_messy_dates = int(0.50 * len(df_messy))

date_indices = np.random.choice(
    df_messy.index,
    size=n_messy_dates,
    replace=False
)

# Introduce inconsistent date formats
for idx in date_indices:
    date_value = pd.to_datetime(
        df_messy.loc[idx, "Application_Date"],
        errors="coerce"
    )

    if pd.notna(date_value):
        format_type = np.random.choice([1, 2, 3, 4])

        if format_type == 1:
            messy_date = date_value.strftime("%d/%m/%Y")
        elif format_type == 2:
            messy_date = date_value.strftime("%Y-%m-%d")
        elif format_type == 3:
            messy_date = date_value.strftime("%b %d, %Y")
        else:
            messy_date = date_value.strftime("%d-%b-%Y")

        df_messy.loc[idx, "Application_Date"] = messy_date

In [17]:
# Check if applied
print(df_messy["Application_Date"].head(20))

0       2026-01-02
1      28-Apr-2026
2      16-Jun-2026
3       2026-03-07
4       2026-04-25
5      21-May-2026
6       2026-07-01
7       2026-04-20
8       2026-08-03
9     Dec 31, 2025
10      2026-04-16
11      2026-01-09
12      2026-03-23
13      2026-04-16
14      2025-10-29
15      2026-03-03
16      2026-08-30
17      2026-05-30
18      2026-07-28
19      30/10/2025
Name: Application_Date, dtype: object


**Missingness and Inconsistent formats of Status Column**

In [18]:

n_missing_statuses = int(0.15 * len(df_messy))

missing_status_indices = np.random.choice(
    df_messy.index,
    size=n_missing_statuses,
    replace=False
)

df_messy.loc[missing_status_indices, "Status"] = np.nan

In [19]:

# Define alternative versions of each status
status_variations = {
    "Under Review": ["under review", "UNDER REVIEW", "Under review"],
    "Selected": ["selected", "SELECTED", "sElEcTeD"],
    "Rejected": ["rejected", "REJECTED", "rejected"]
}

# Select 30% of rows
n_messy_statuses = int(0.30 * len(df_messy))

status_indices = np.random.choice(
    df_messy.index,
    size=n_messy_statuses,
    replace=False
)

# Introduce inconsistent capitalization
for idx in status_indices:
    status = df_messy.loc[idx, "Status"]

    if pd.notna(status) and status in status_variations:
        df_messy.loc[idx, "Status"] = np.random.choice(
            status_variations[status]
        )

In [20]:

print(df_messy["Status"].value_counts(dropna=False))

Status
Selected        216
Rejected        207
Under Review    173
NaN             150
rejected         60
SELECTED         34
under review     32
Under review     32
UNDER REVIEW     30
selected         23
sElEcTeD         22
REJECTED         21
Name: count, dtype: int64


In [21]:
# Creating messy data
df_messy.to_csv("messy_applicants.csv", index=False)